# AmazonHelp AI Support Agent

This notebook contains the complete development and evaluation workflow for an AI customer-support agent built from historical AmazonHelp conversations on Twitter.

The system performs three main tasks:

1. **Intent classification** — classify an incoming customer message into one of 11 support intents.
2. **Historical retrieval + response generation** — retrieve similar AmazonHelp conversations and use their historical responses as grounding evidence for a generated reply.
3. **Escalation decision** — decide whether the request can be safely handled automatically or should be escalated to a human, with a reason.

The project uses the **Customer Support on Twitter** dataset and focuses on the `AmazonHelp` brand.

The notebook covers:

- exploratory data analysis and brand selection;
- construction of customer → AmazonHelp response pairs;
- text cleaning and English-language filtering;

`https://colab.research.google.com/drive/1dzpz4EoXlFiRx0h31VZa9MgJsNngiP-d?usp=sharing`


In [2]:
import os
import re
import html
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

RANDOM_STATE = 42

warnings.filterwarnings("ignore")

In [6]:
df = pd.read_csv("/content/twcs.csv")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (2811774, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [7]:
# Basic Dataset Inspection

print("Shape:", df.shape)

display(df.head())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

Shape: (2811774, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0



Data types:


,0
tweet_id,int64
author_id,object
inbound,bool
created_at,object
text,object
response_tweet_id,object
in_response_to_tweet_id,float64



Missing values:


,0
tweet_id,0
author_id,0
inbound,0
created_at,0
text,0
response_tweet_id,1040629
in_response_to_tweet_id,794335


In [8]:
print("Inbound value counts:")
display(df["inbound"].value_counts(dropna=False))

print("\nresponse_tweet_id coverage:")
print(df["response_tweet_id"].notna().sum(), "non-null")
print(df["response_tweet_id"].isna().sum(), "null")

print("\nin_response_to_tweet_id coverage:")
print(df["in_response_to_tweet_id"].notna().sum(), "non-null")
print(df["in_response_to_tweet_id"].isna().sum(), "null")

Inbound value counts:


,count
inbound,
True,1537843
False,1273931



response_tweet_id coverage:
1771145 non-null
1040629 null

in_response_to_tweet_id coverage:
2017439 non-null
794335 null


In [9]:
# Identify Support Brands

brand_counts = (
    df.loc[df["inbound"] == False, "author_id"]
      .value_counts()
)

print("Number of potential brand/support accounts:", len(brand_counts))

display(brand_counts.head(20))

Number of potential brand/support accounts: 108


,count
author_id,
AmazonHelp,169840
AppleSupport,106860
Uber_Support,56270
SpotifyCares,43265
Delta,42253
Tesco,38573
AmericanAir,36764
TMobileHelp,34317
comcastcares,33031


In [10]:
# Inspect Brand / Support Accounts


brand_tweets = df[df["inbound"] == False].copy()

print("Sample brand tweets:")
display(
    brand_tweets[
        ["author_id", "text", "response_tweet_id", "in_response_to_tweet_id"]
    ].head(20)
)

Sample brand tweets:


,author_id,text,response_tweet_id,in_response_to_tweet_id
0,sprintcare,@115712 I understand. I would like to assist y...,2,3.0
3,sprintcare,@115712 Please send us a Private Message so th...,3,5.0
5,sprintcare,@115712 Can you please send us a private messa...,"5,7",8.0
7,sprintcare,@115713 This is saddening to hear. Please shoo...,NaN,12.0
9,sprintcare,@115713 We understand your concerns and we'd l...,12,16.0
11,sprintcare,@115713 H there! We'd definitely like to work ...,16,18.0
13,sprintcare,@115715 Please send me a private message so th...,NaN,20.0
15,Ask_Spectrum,@115716 What information is incorrect? ^JK,"22,23",24.0
17,Ask_Spectrum,@115716 Our department is part of the corporat...,26,22.0
19,Ask_Spectrum,@115716 No thank you. ^JK,NaN,26.0


In [12]:
# Select AmazonHelp

brand = "AmazonHelp"

amazon = df[df["author_id"] == brand].copy()

print("Total AmazonHelp tweets:", len(amazon))

print("\nInbound / Outbound:")
print(amazon["inbound"].value_counts())

print("\nTweets with a response:")
print(amazon["response_tweet_id"].notna().sum())

print("\nTweets responding to another tweet:")
print(amazon["in_response_to_tweet_id"].notna().sum())

Total AmazonHelp tweets: 169840

Inbound / Outbound:
inbound
False    169840
Name: count, dtype: int64

Tweets with a response:
85274

Tweets responding to another tweet:
169287


In [14]:
# Inspect an AmazonHelp Response Relationship

amazon = df[df["author_id"] == "AmazonHelp"].copy()

# Select an AmazonHelp tweet that is a response to another tweet
example = amazon[
    amazon["in_response_to_tweet_id"].notna()
].iloc[0]

print("AmazonHelp tweet:")
display(
    example[
        [
            "tweet_id",
            "author_id",
            "inbound",
            "text",
            "in_response_to_tweet_id"
        ]
    ].to_frame().T
)

AmazonHelp tweet:


,tweet_id,author_id,inbound,text,in_response_to_tweet_id
181,269,AmazonHelp,False,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,272.0


In [17]:
# Construct Customer → AmazonHelp Response Pairs


tweet_lookup = df.set_index("tweet_id")

pairs = []

for _, amazon_row in amazon.iterrows():

    parent_id = amazon_row["in_response_to_tweet_id"]

    if pd.isna(parent_id):
        continue

    if parent_id not in tweet_lookup.index:
        continue

    customer_row = tweet_lookup.loc[parent_id]

    # Only keep direct customer → AmazonHelp interactions
    if customer_row["inbound"] != True:
        continue

    pairs.append({
        "customer_tweet_id": customer_row.name,
        "customer_id": customer_row["author_id"],
        "customer_created_at": customer_row["created_at"],
        "customer_message": customer_row["text"],
        "amazon_tweet_id": amazon_row["tweet_id"],
        "amazon_created_at": amazon_row["created_at"],
        "amazon_response": amazon_row["text"]
    })

pairs_df = pd.DataFrame(pairs)

print("Customer → AmazonHelp pairs:", len(pairs_df))
display(pairs_df.head())

Customer → AmazonHelp pairs: 168814


,customer_tweet_id,customer_id,customer_created_at,customer_message,amazon_tweet_id,amazon_created_at,amazon_response
0,272,115770,Wed Nov 22 09:14:39 +0000 2017,amazonのfireTVstickが見れない😢,269,Wed Nov 22 09:23:01 +0000 2017,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
1,271,115770,Wed Nov 22 09:30:36 +0000 2017,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,273,Wed Nov 22 09:40:27 +0000 2017,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
2,274,115770,Wed Nov 22 09:44:04 +0000 2017,@AmazonHelp こちらこそありがとうございました。,275,Wed Nov 22 10:06:26 +0000 2017,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
3,325,115792,Wed Nov 22 08:55:35 +0000 2017,amazonプライムビデオ、再生エラーが多いです,324,Wed Nov 22 09:06:00 +0000 2017,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
4,617,115820,Tue Oct 31 22:16:32 +0000 2017,Way to drop the ball on customer service @1158...,615,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...


In [18]:
# Inspect Constructed Customer → AmazonHelp Pairs

print("Pairs shape:", pairs_df.shape)

print("\nColumns:")
print(pairs_df.columns.tolist())

print("\nMissing values:")
display(pairs_df.isna().sum())

print("\nSample conversations:")
display(
    pairs_df[
        ["customer_message", "amazon_response"]
    ].head(10)
)

Pairs shape: (168814, 7)

Columns:
['customer_tweet_id', 'customer_id', 'customer_created_at', 'customer_message', 'amazon_tweet_id', 'amazon_created_at', 'amazon_response']

Missing values:


,0
customer_tweet_id,0
customer_id,0
customer_created_at,0
customer_message,0
amazon_tweet_id,0
amazon_created_at,0
amazon_response,0



Sample conversations:


,customer_message,amazon_response
0,amazonのfireTVstickが見れない😢,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
1,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
2,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
3,amazonプライムビデオ、再生エラーが多いです,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
4,Way to drop the ball on customer service @1158...,@115820 I'm sorry we've let you down! Without ...
5,@AmazonHelp 3 different people have given 3 di...,@115820 We'd like to take a further look into ...
6,@115823 I want my amazon payments account CLOS...,@115822 I am unable to affect your account via...
7,"@115825 also, beim Addams Family-Film in Prime...","@115824 Hi, wir erhalten die Filme/Serien so v..."
8,"@AmazonHelp Okay, danke für die Info",@115824 Wir haben zu danken. Schönen Abend noc...
9,@115828 How about you guys figure out my Xbox ...,@115826 I'm sorry for the wait. You'll receive...


In [20]:
# ============================================================
# Text Cleaning Function
# ============================================================

def clean_text(text):
    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove Twitter @mentions
    text = re.sub(r"@\w+", " ", text)

    # Remove Twitter artifacts such as ^AB
    text = re.sub(r"\^[A-Z]{2}\b", " ", text)

    # Remove hashtag symbol but preserve the word
    text = re.sub(r"#(\w+)", r"\1", text)

    # Keep letters, numbers and whitespace
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Lowercase
    text = text.lower()

    return text


In [21]:
# Clean Customer Messages and AmazonHelp Responses

pairs_df["cleaned_customer_message"] = (
    pairs_df["customer_message"].apply(clean_text)
)

pairs_df["cleaned_amazon_response"] = (
    pairs_df["amazon_response"].apply(clean_text)
)

print("Cleaning complete.")

display(
    pairs_df[
        [
            "customer_message",
            "cleaned_customer_message",
            "amazon_response",
            "cleaned_amazon_response"
        ]
    ].head(10)
)

Cleaning complete.


,customer_message,cleaned_customer_message,amazon_response,cleaned_amazon_response
0,amazonのfireTVstickが見れない😢,amazon firetvstick,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,fire tv stick et
1,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,et
2,@AmazonHelp こちらこそありがとうございました。,,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,et
3,amazonプライムビデオ、再生エラーが多いです,amazon,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,et
4,Way to drop the ball on customer service @1158...,way to drop the ball on customer service so pi...,@115820 I'm sorry we've let you down! Without ...,i m sorry we ve let you down without providing...
5,@AmazonHelp 3 different people have given 3 di...,3 different people have given 3 different answ...,@115820 We'd like to take a further look into ...,we d like to take a further look into this wit...
6,@115823 I want my amazon payments account CLOS...,i want my amazon payments account closed dm me...,@115822 I am unable to affect your account via...,i am unable to affect your account via twitter...
7,"@115825 also, beim Addams Family-Film in Prime...",also beim addams family film in prime sind bil...,"@115824 Hi, wir erhalten die Filme/Serien so v...",hi wir erhalten die filme serien so vom jeweil...
8,"@AmazonHelp Okay, danke für die Info",okay danke f r die info,@115824 Wir haben zu danken. Schönen Abend noc...,wir haben zu danken sch nen abend noch
9,@115828 How about you guys figure out my Xbox ...,how about you guys figure out my xbox one x pr...,@115826 I'm sorry for the wait. You'll receive...,i m sorry for the wait you ll receive an email...


In [22]:
# Remove Empty / Unusable Rows

before = len(pairs_df)

pairs_df = pairs_df[
    (pairs_df["cleaned_customer_message"].str.len() > 0) &
    (pairs_df["cleaned_amazon_response"].str.len() > 0)
].copy()

after = len(pairs_df)

print("Rows before removing empty text:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before removing empty text: 168814
Rows after: 163348
Rows removed: 5466


In [23]:

# Remove Exact Duplicate Customer → Response Pairs

before = len(pairs_df)

pairs_df = pairs_df.drop_duplicates(
    subset=[
        "cleaned_customer_message",
        "cleaned_amazon_response"
    ]
).reset_index(drop=True)

after = len(pairs_df)

print("Rows before deduplication:", before)
print("Rows after deduplication:", after)
print("Duplicate pairs removed:", before - after)

Rows before deduplication: 163348
Rows after deduplication: 159945
Duplicate pairs removed: 3403


In [26]:
!pip install langid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 49.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langid: filename=langid-1.1.6-py3-none-any.whl size=1941216 sha256=0c8461a28c33d33ba8f05b7e539d3b6ec155f907967671515d44673c1b499254
  Stored in directory: /root/.cache/pip/wheels/50/d7/8b/f20e951da531c61b96b87311b48dd1cc6fedaf1e37c581aaee
Successfully built langid


In [28]:
import langid
from tqdm.auto import tqdm

tqdm.pandas()

# Detect language of each customer message
pairs_df["language"] = pairs_df["customer_message"].progress_apply(
    lambda x: langid.classify(str(x))[0]
)

# Keep English customer messages
english_pairs = pairs_df[
    pairs_df["language"] == "en"
].copy()

english_pairs = english_pairs.reset_index(drop=True)

print("Original pairs:", len(pairs_df))
print("English pairs:", len(english_pairs))
print(
    "English percentage:",
    round(len(english_pairs) / len(pairs_df) * 100, 2),
    "%"
)

  0%|          | 0/159945 [00:00<?, ?it/s]

Original pairs: 159945
English pairs: 128448
English percentage: 80.31 %


In [31]:
# Inspect English Dataset

print("English pairs shape:", english_pairs.shape)

print("\nSample English conversations:")
display(
    english_pairs[
        ["cleaned_customer_message", "cleaned_amazon_response"]
    ].head(10)
)

print("\nLanguage distribution:")
display(
    english_pairs["language"].value_counts()
)


English pairs shape: (128448, 11)

Sample English conversations:


,cleaned_customer_message,cleaned_amazon_response
0,way to drop the ball on customer service so pi...,i m sorry we ve let you down without providing...
1,3 different people have given 3 different answ...,we d like to take a further look into this wit...
2,i want my amazon payments account closed dm me...,i am unable to affect your account via twitter...
3,how about you guys figure out my xbox one x pr...,i m sorry for the wait you ll receive an email...
4,yeah this is crazy we re less than a week away...,thanks for your patience
5,my package was accidentally opened 4 items mis...,i m sorry your order arrived in this condition...
6,why is my order at my local courier for the la...,i m sorry for the wait please reach out to us ...
7,thanks for the style advice look i think hallo...,alexa says both styles are working for you my ...
8,hi ready for some help,were you able to reach us at the link dw provided
9,is the echo show no longer supported,the echo show is supported please reach us for...



Language distribution:


,count
language,
en,128448


In [30]:
# ============================================================
# Verify Cleaning on English Conversations
# ============================================================

print("Original customer messages:")
display(
    english_pairs[
        ["customer_message", "cleaned_customer_message"]
    ].head(10)
)

print("\nOriginal AmazonHelp responses:")
display(
    english_pairs[
        ["amazon_response", "cleaned_amazon_response"]
    ].head(10)
)

Original customer messages:


,customer_message,cleaned_customer_message
0,Way to drop the ball on customer service @1158...,way to drop the ball on customer service so pi...
1,@AmazonHelp 3 different people have given 3 di...,3 different people have given 3 different answ...
2,@115823 I want my amazon payments account CLOS...,i want my amazon payments account closed dm me...
3,@115828 How about you guys figure out my Xbox ...,how about you guys figure out my xbox one x pr...
4,@AmazonHelp @115826 Yeah this is crazy we’re l...,yeah this is crazy we re less than a week away...
5,@115830 my package was ‘accidentally’ opened.....,my package was accidentally opened 4 items mis...
6,@115821 @AmazonHelp why is my order at my loca...,why is my order at my local courier for the la...
7,"Thanks for the style advice, @115833 look ...I...",thanks for the style advice look i think hallo...
8,@AmazonHelp Hi ready for some help,hi ready for some help
9,@AmazonHelp Is the Echo Show no longer supported?,is the echo show no longer supported



Original AmazonHelp responses:


,amazon_response,cleaned_amazon_response
0,@115820 I'm sorry we've let you down! Without ...,i m sorry we ve let you down without providing...
1,@115820 We'd like to take a further look into ...,we d like to take a further look into this wit...
2,@115822 I am unable to affect your account via...,i am unable to affect your account via twitter...
3,@115826 I'm sorry for the wait. You'll receive...,i m sorry for the wait you ll receive an email...
4,@115827 Thanks for your patience. ^KM,thanks for your patience
5,@115829 I'm sorry your order arrived in this c...,i m sorry your order arrived in this condition...
6,@115831 I'm sorry for the wait. Please reach o...,i m sorry for the wait please reach out to us ...
7,@115832 Alexa says both styles are working for...,alexa says both styles are working for you my ...
8,@115834 Were you able to reach us at the link ...,were you able to reach us at the link dw provided
9,"@115834 The Echo Show is supported, please rea...",the echo show is supported please reach us for...


In [32]:
# Create and Save Final Cleaned Dataset

final_df = english_pairs[
    [
        "cleaned_customer_message",
        "cleaned_amazon_response"
    ]
].copy()

final_df = final_df[
    (final_df["cleaned_customer_message"].str.strip() != "") &
    (final_df["cleaned_amazon_response"].str.strip() != "")
].copy()

# Remove exact duplicate customer-response pairs
final_df = final_df.drop_duplicates(
    subset=[
        "cleaned_customer_message",
        "cleaned_amazon_response"
    ]
).reset_index(drop=True)


# Save
FINAL_DATA_PATH = "amazonhelp_cleaned_with_intents.csv"

final_df.to_csv(
    FINAL_DATA_PATH,
    index=False
)

print(f"Saved: {FINAL_DATA_PATH}")
print(f"Rows: {len(final_df):,}")
print(f"Columns: {final_df.columns.tolist()}")

Saved: amazonhelp_cleaned_with_intents.csv
Rows: 128,448
Columns: ['cleaned_customer_message', 'cleaned_amazon_response']
